# 🏁 28 · Capstone: A Complete Bioimage-Analysis Pipeline
### From raw pixels to a checked biological answer — and three ways to take it further

**Part K · Capstone** · Module 28 of 29 · builds on all

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/28_capstone_pipeline.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (Colab free tier); every step finishes in well under the estimated time below |
| **Core frameworks** | NumPy, SciPy, scikit-image, PyTorch, Matplotlib |
| **Prerequisites** | The whole course — this capstone deliberately reuses tools from Modules 00 through 27 |
| **02-740 topics covered** | Restoration, segmentation, detection, tracking, and spatial statistics — integrated into one end-to-end pipeline |
| **Builds on** | all |
| **Estimated time** | 150–200 min |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED.** The multi-channel, multi-timepoint tissue this capstone analyzes, and every ground-truth label used to check the pipeline's output, are generated by code — so every number this notebook reports at the end can be checked against a known right answer, not just inspected by eye.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Build one complete bioimage-analysis pipeline from scratch — restore, segment, detect/classify,
  track, and quantify spatially — chaining real functions from this course's own toolkit rather
  than a sketch of what each step would do.
- Check every stage of that pipeline against known ground truth, and report a single, honest,
  checked results summary at the end rather than a collection of disconnected demos.
- Choose and scope your own extension of the pipeline from three concrete project tracks spanning
  classical computer vision, deep learning, and this course's research-frontier modules.
- Apply a reproducibility checklist — seeded randomness, honest reporting of what is simulated,
  awareness of train/test leakage and dataset shift — to your own future bioimage-analysis work.

## 🗺️ Contents

1. The pipeline, end to end: simulate a multi-channel time-lapse tissue
2. Restore: recovering a known degradation
3. Segment: classical watershed instance segmentation, checked against ground truth
4. Detect & classify: is each cell marker-positive, checked against ground truth
5. Track: linking detections across time with the Hungarian algorithm
6. Quantify: spatial statistics and colocalization
7. Report: one checked results summary
8. Project studio: three extension tracks
9. Reproducibility checklist
10. 📘 Lecture notes — the whole course's throughline

## 🧫 The biological question

Put yourself in the position this whole course has been training you for: a collaborator hands
you a time-lapse movie of a developing tissue, imaged in two fluorescence channels — one marking
every nucleus, the other a reporter expressed only in some cells — and asks a genuinely biological
question: *do the reporter-positive cells end up clustered together, or scattered randomly, and
does their reporter signal actually track with their nuclear-channel identity over time?*
Answering this for real requires every stage of the pipeline this course has built, piece by
piece, module by module: restoring a degraded image, segmenting cells, deciding what kind of cell
each one is, following cells across time, and quantifying the result spatially — and, at every
step, checking that step's output before trusting the next one with it (Module 26's lesson, now
applied end to end rather than to one isolated demo).

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 28
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

The exact same helper functions used throughout this course, copied verbatim so this notebook runs on its own.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_time_lapse(n_frames: int = 20, n_particles: int = 12, size: int = 160,
                         speed: float = 2.5, division_prob: float = 0.03,
                         seed: int = 0):
    """A SIMULATED time-lapse of moving, occasionally dividing particles/cells.

    Returns ``frames`` (list of n_frames images), ``tracks`` (dict: true track id -> list of
    (frame, y, x)), and ``lineage`` (dict: child track id -> parent track id, for divisions).
    """
    rng = np.random.default_rng(seed)
    margin = size * 0.1
    positions = {i: np.array([rng.uniform(margin, size - margin), rng.uniform(margin, size - margin)])
                 for i in range(n_particles)}
    velocities = {i: rng.normal(0, speed, size=2) for i in positions}
    tracks = {i: [] for i in positions}
    lineage = {}
    next_id = n_particles
    frames = []
    for t in range(n_frames):
        canvas = np.full((size, size), 0.05)
        for pid in list(positions):
            positions[pid] = np.clip(positions[pid] + velocities[pid], margin, size - margin)
            for k in (0, 1):
                if positions[pid][k] in (margin, size - margin):
                    velocities[pid][k] *= -1
            _draw_blob(canvas, *positions[pid], size * 0.03, "round", rng)
            tracks[pid].append((t, *positions[pid]))
            if t > 2 and rng.random() < division_prob and len(positions) < n_particles * 3:
                child = next_id
                next_id += 1
                positions[child] = positions[pid] + rng.normal(0, 3, size=2)
                velocities[child] = -velocities[pid] + rng.normal(0, speed * 0.3, size=2)
                tracks[child] = []
                lineage[child] = pid
        canvas = blur_image(canvas, gaussian_psf(1.1))
        frames.append(add_gaussian_noise(canvas, 0.02, rng).clip(0, None))
    return dict(frames=frames, tracks=tracks, lineage=lineage, source="SIMULATED")


SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def degrade_image(image: np.ndarray, psf_sigma: float = 1.5, noise_sigma: float = 0.04,
                   seed: int = 0) -> dict:
    """Blur + noise a clean image with a KNOWN point-spread function (the forward model of
    restoration/deconvolution). Returns ``degraded`` and the ``psf`` used."""
    rng = np.random.default_rng(seed)
    psf = gaussian_psf(psf_sigma)
    degraded = blur_image(image, psf)
    degraded = add_gaussian_noise(degraded, noise_sigma, rng).clip(0, None)
    return dict(degraded=degraded, psf=psf, psf_sigma=psf_sigma, noise_sigma=noise_sigma)


def wiener_deconvolve(degraded: np.ndarray, psf: np.ndarray, noise_to_signal: float = 0.01) -> np.ndarray:
    """Wiener deconvolution: inverse-filter a blurred, noisy image given its (known) PSF and an
    estimated noise-to-signal power ratio, which regularises the inverse near-zero frequencies."""
    h, w = degraded.shape
    psf_padded = np.zeros((h, w))
    ph, pw = psf.shape
    psf_padded[:ph, :pw] = psf
    psf_padded = np.roll(psf_padded, (-ph // 2, -pw // 2), axis=(0, 1))
    H = np.fft.fft2(psf_padded)
    G = np.fft.fft2(degraded)
    Hc = np.conj(H)
    wiener = Hc / (H * Hc + noise_to_signal)
    return np.real(np.fft.ifft2(wiener * G))


def psnr(reference: np.ndarray, estimate: np.ndarray, data_range: float | None = None) -> float:
    """Peak signal-to-noise ratio (dB) between a reference and an estimate; higher is better."""
    data_range = data_range or (reference.max() - reference.min() + 1e-9)
    mse = np.mean((reference - estimate) ** 2)
    if mse <= 1e-12:
        return float("inf")
    return 10 * np.log10(data_range ** 2 / mse)


def watershed_labels(image: np.ndarray, min_distance: int = 7) -> np.ndarray:
    """Marker-controlled watershed segmentation of a blobby intensity image: finds markers at
    local intensity maxima (one per expected object) and floods from them. Returns an integer
    label image (0 = background)."""
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed as _watershed
    smooth = ndi.gaussian_filter(image, 1.0)
    coords = peak_local_max(smooth, min_distance=min_distance, labels=smooth > smooth.mean())
    markers = np.zeros(image.shape, dtype=int)
    for i, (y, x) in enumerate(coords, start=1):
        markers[y, x] = i
    markers = ndi.label(ndi.binary_dilation(markers > 0, iterations=2))[0] if coords.size else markers
    elevation = -smooth
    return _watershed(elevation, markers=markers, mask=smooth > smooth.mean() * 0.6)


def shape_features(mask: np.ndarray) -> dict:
    """Simple morphological/shape features of a binary blob mask: area, perimeter (boundary
    pixel count), equivalent-circle compactness, and the aspect ratio of its bounding box."""
    area = mask.sum()
    eroded = ndi.binary_erosion(mask)
    perimeter = max(1, (mask & ~eroded).sum())
    compactness = 4 * np.pi * area / (perimeter ** 2 + 1e-9)  # 1.0 for a perfect circle
    ys, xs = np.nonzero(mask)
    if len(ys) == 0:
        return dict(area=0, perimeter=0, compactness=0.0, aspect_ratio=1.0)
    h = ys.max() - ys.min() + 1
    w = xs.max() - xs.min() + 1
    aspect_ratio = max(h, w) / max(1, min(h, w))
    return dict(area=float(area), perimeter=float(perimeter),
                compactness=float(compactness), aspect_ratio=float(aspect_ratio))


def hungarian_assignment(cost_matrix: np.ndarray):
    """The Hungarian algorithm for the linear assignment problem: the lowest-total-cost one-to-
    one matching between rows and columns of ``cost_matrix``. Returns (row_ind, col_ind)."""
    return linear_sum_assignment(cost_matrix)


def dice_coefficient(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Dice/F1 overlap of two binary masks: 2|A∩B| / (|A|+|B|)."""
    a, b = mask_a.astype(bool), mask_b.astype(bool)
    inter = (a & b).sum()
    denom = a.sum() + b.sum()
    return float(2 * inter / denom) if denom > 0 else 1.0


def nearest_neighbor_distances(points: np.ndarray, other: np.ndarray | None = None) -> np.ndarray:
    """For each point, the distance to its nearest neighbour (in ``other`` if given — a CROSS
    nearest-neighbour distance between two populations — else in ``points`` itself)."""
    target = points if other is None else other
    d = np.sqrt(((points[:, None, :] - target[None, :, :]) ** 2).sum(axis=2))
    if other is None:
        np.fill_diagonal(d, np.inf)
    return d.min(axis=1)


def clark_evans_index(points: np.ndarray, area: float) -> float:
    """The Clark–Evans index: observed mean nearest-neighbour distance divided by the distance
    expected under complete spatial randomness (CSR) at the same point density. CE < 1 means
    clustering, CE ≈ 1 random, CE > 1 dispersion/regularity."""
    n = len(points)
    observed = nearest_neighbor_distances(points).mean()
    density = n / area
    expected = 1.0 / (2 * np.sqrt(density))
    return float(observed / expected)


def pearson_colocalization(ch_a: np.ndarray, ch_b: np.ndarray, mask: np.ndarray | None = None) -> float:
    """Pearson's colocalization coefficient: the ordinary linear correlation between two
    channels' pixel intensities, restricted to ``mask`` if given (e.g. just the cell interior)."""
    a = ch_a[mask] if mask is not None else ch_a.ravel()
    b = ch_b[mask] if mask is not None else ch_b.ravel()
    a, b = a - a.mean(), b - b.mean()
    denom = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / denom) if denom > 0 else 0.0


def manders_coefficients(ch_a: np.ndarray, ch_b: np.ndarray, thr_a: float, thr_b: float):
    """Manders' colocalization coefficients M1, M2 at the given intensity thresholds: the
    fraction of channel A's (resp. B's) total signal that sits in pixels where the OTHER
    channel is also above its threshold."""
    pos_b = ch_b > thr_b
    pos_a = ch_a > thr_a
    m1 = ch_a[pos_b].sum() / (ch_a.sum() + 1e-9)
    m2 = ch_b[pos_a].sum() / (ch_b.sum() + 1e-9)
    return float(m1), float(m2)


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes


def plot_tracks(ax, tracks: dict, min_len: int = 2):
    """Plot every track in a ``{id: [(t, y, x), ...]}`` dict as a coloured polyline."""
    for tid, pts in tracks.items():
        if len(pts) < min_len:
            continue
        arr = np.array(pts)
        ax.plot(arr[:, 2], arr[:, 1], marker="o", markersize=2, linewidth=1)
    ax.invert_yaxis()
    ax.set_aspect("equal")

---

## 1 · The pipeline, end to end: simulate a multi-channel time-lapse tissue

We build the "raw data" for this capstone the same way every other dataset in this course was
built: entirely by code, with every label we will later check against kept on hand. Channel A is
a nuclear marker present in every cell (built with `simulate_time_lapse`, exactly as in
Module 14); channel B is a reporter expressed only in a known subset of the tracked cells — the
ground truth this capstone's classification step will be checked against.

In [ ]:
# ── Channel A: every tracked cell, over time (exactly Module 14's simulator) ──
N_FRAMES, SIZE = 14, 160
sim_time = simulate_time_lapse(n_frames=N_FRAMES, n_particles=14, size=SIZE, speed=2.0,
                                division_prob=0.05, seed=SEED)
frames_a, tracks, lineage = sim_time["frames"], sim_time["tracks"], sim_time["lineage"]
print(f"{N_FRAMES} frames, {len(tracks)} total tracked objects "
      f"({sum(1 for c in lineage)} arose from a division)")


def active_positions_at(tracks, t):
    """{track_id: (y, x)} for every track alive at frame t."""
    pos = {}
    for tid, pts in tracks.items():
        for (ft, y, x) in pts:
            if ft == t:
                pos[tid] = (y, x)
    return pos


print(f"objects active in frame 0 : {len(active_positions_at(tracks, 0))}")
print(f"objects active in frame {N_FRAMES-1} : {len(active_positions_at(tracks, N_FRAMES-1))}")

In [ ]:
# ── Channel B: a reporter expressed only by a known subset of tracked cells ──
MARKER_POSITIVE = {tid for tid in tracks if tid % 2 == 0}   # the (known!) ground-truth rule


def simulate_marker_channel(tracks, n_frames, size, positive_ids, psf_sigma=1.1, noise_sigma=0.02,
                             seed=0):
    """A SIMULATED second fluorescence channel: cells in `positive_ids` render a reporter blob at
    their (known, tracked) position every frame they are alive; others render nothing."""
    rng_local = np.random.default_rng(seed)
    frames = []
    for t in range(n_frames):
        canvas = np.zeros((size, size))
        for tid, (y, x) in active_positions_at(tracks, t).items():
            if tid in positive_ids:
                _draw_blob(canvas, y, x, size * 0.03, "round", rng_local)
        canvas = blur_image(canvas, gaussian_psf(psf_sigma))
        frames.append(np.clip(add_gaussian_noise(canvas, noise_sigma, rng_local), 0, None))
    return frames


frames_b = simulate_marker_channel(tracks, N_FRAMES, SIZE, MARKER_POSITIVE, seed=SEED + 1)
print(f"{len(MARKER_POSITIVE)} of {len(tracks)} tracked cells are reporter-positive (ground truth)")

fig, axes = plt.subplots(1, 2, figsize=(7, 3.4))
show_image(axes[0], frames_a[0], "channel A: nuclear marker")
show_image(axes[1], frames_b[0], "channel B: reporter (positive cells only)")
fig.suptitle("Frame 0")
fig.tight_layout()
plt.show()

🔍 **What just happened?** We now have two aligned channels across 14 frames, plus
— because we built it ourselves — complete ground truth: exactly which cells exist at every
frame (`tracks`), exactly which divided and from whom (`lineage`), and exactly which cells should
read out as reporter-positive (`MARKER_POSITIVE`). Every check in the rest of this notebook
compares a pipeline stage's output against one of these three ground-truth sources.

---

## 2 · Restore: recovering a known degradation

Real time-lapse imaging often degrades mid-acquisition — a focus drift, a dirtier objective, a
dimmer laser late in a long movie. We simulate exactly this on one representative frame using
`degrade_image` (Module 17's forward model: a known point-spread function plus noise) and recover
it with Wiener deconvolution, the same restoration this course taught in Module 17.

In [ ]:
# ── 🧪 Degrade one frame with a KNOWN point-spread function, then restore it ──
ref_frame = frames_a[N_FRAMES // 2]
degraded = degrade_image(ref_frame, psf_sigma=2.2, noise_sigma=0.05, seed=SEED)
restored = wiener_deconvolve(degraded["degraded"], degraded["psf"], noise_to_signal=0.02)

psnr_before = psnr(ref_frame, degraded["degraded"])
psnr_after = psnr(ref_frame, restored)
print(f"PSNR, degraded frame vs. reference : {psnr_before:.1f} dB")
print(f"PSNR, restored frame vs. reference  : {psnr_after:.1f} dB  "
      f"({psnr_after - psnr_before:+.1f} dB)")

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
show_image(axes[0], ref_frame, "reference (pre-degradation)")
show_image(axes[1], degraded["degraded"], f"degraded ({psnr_before:.1f} dB)")
show_image(axes[2], restored, f"restored ({psnr_after:.1f} dB)")
fig.tight_layout()
plt.show()

🧪 **Ground-truth check.** `ref_frame` is the exact pre-degradation image, so the PSNR
gain above is not a judgment call — it is a direct, checkable measurement of how much of the known
degradation Wiener deconvolution recovered. From here on, the pipeline proceeds on the *restored*
frame for this one timepoint and the original (mildly noisy but undegraded) frames for the rest —
exactly the situation a real pipeline faces when only part of an acquisition needs restoring.

---

## 3 · Segment: classical watershed instance segmentation, checked against ground truth

We segment every frame's channel A with the classical watershed pipeline from Module 10, and
check the result against the exact instance mask we can reconstruct from `tracks` (since we know
precisely where every cell was placed). A cell in this movie occupies only a few percent of the
frame — mostly background — so, exactly as a real pipeline would, we first subtract a robust
background estimate (a high percentile of the frame's own intensities) before segmenting; skipping
this step leaves `watershed_labels`' fixed relative threshold badly miscalibrated for an image
this background-dominated, and it segments almost the whole frame as foreground.

In [ ]:
# ── 🧪 Segment every frame; check against a ground-truth mask rebuilt from `tracks` ──
def ground_truth_mask_at(tracks, t, size):
    """The exact instance mask implied by `tracks` at frame t (same blob radius/shape the
    simulator itself used), for checking a segmentation against ground truth."""
    label_img = np.zeros((size, size), dtype=int)
    for i, (tid, (y, x)) in enumerate(sorted(active_positions_at(tracks, t).items()), start=1):
        tmp = np.zeros((size, size))
        _draw_blob(tmp, y, x, size * 0.03, "round", np.random.default_rng(0))
        label_img[(tmp > 0.25) & (label_img == 0)] = i
    return label_img


def subtract_background(image, percentile=98.5):
    """A simple, robust background estimate: a high percentile of the image's own pixel values
    (most pixels are background in a sparse field of small cells), subtracted and clipped."""
    return np.clip(image - np.percentile(image, percentile), 0, None)


pred_labels, gt_counts, pred_counts, dices = [], [], [], []
for t in range(N_FRAMES):
    frame = restored if t == N_FRAMES // 2 else frames_a[t]
    labels = watershed_labels(subtract_background(frame))
    gt_mask = ground_truth_mask_at(tracks, t, SIZE)
    pred_labels.append(labels)
    gt_counts.append(gt_mask.max())
    pred_counts.append(labels.max())
    dices.append(dice_coefficient(labels > 0, gt_mask > 0))

print(f"mean object count, ground truth : {np.mean(gt_counts):.1f}")
print(f"mean object count, segmented    : {np.mean(pred_counts):.1f}")
print(f"mean Dice (segmented vs. exact ground-truth mask), across {N_FRAMES} frames: "
      f"{np.mean(dices):.3f}")

🧪 **Ground-truth check.** `ground_truth_mask_at` is built from the exact positions
`tracks` says every cell occupied, using the same blob shape the simulator itself drew — so the
mean Dice above is an honest, per-frame measurement of segmentation quality, not an eyeballed
impression from one picture. A Dice score well above the near-zero score a random or empty
segmentation would get confirms the pipeline's first processing stage is trustworthy enough to
build the rest of the pipeline on. Note the object *count* still undershoots the true count
somewhat (a real, honestly reported limitation of a fixed `min_distance` watershed on cells that
sit close together — see Track A in §8 for one way to address it) even though the overlap-based
Dice score is high; count and overlap are different, both useful, metrics.

---

## 4 · Detect & classify: is each cell marker-positive, checked against ground truth

For every segmented object, we read out channel B's mean intensity under its mask and classify it
reporter-positive or -negative by a simple threshold. To check this against ground truth, each
detection is matched to the nearest known track position (we have both coordinates because this
is simulated data) and compared to `MARKER_POSITIVE`.

In [ ]:
# ── 🧪 Classify every detection, matched to ground truth by nearest known position ──
def detection_centroids(labels):
    """(y, x) centroid of every labeled object, by id."""
    out = {}
    for obj_id in range(1, labels.max() + 1):
        ys, xs = np.nonzero(labels == obj_id)
        if len(ys):
            out[obj_id] = (ys.mean(), xs.mean())
    return out


marker_threshold = np.mean([f.mean() for f in frames_b]) * 2.5   # a simple, fixed threshold
classification_records = []   # (frame, predicted_positive, true_positive)
for t in range(N_FRAMES):
    labels = pred_labels[t]
    centroids = detection_centroids(labels)
    true_positions = active_positions_at(tracks, t)   # {tid: (y, x)}
    true_ids = list(true_positions)
    true_coords = np.array([true_positions[tid] for tid in true_ids])
    frame_b = frames_b[t]
    for obj_id, (cy, cx) in centroids.items():
        mask = labels == obj_id
        predicted_positive = frame_b[mask].mean() > marker_threshold
        # match this detection to its nearest TRUE tracked position (ground truth correspondence)
        d = np.hypot(true_coords[:, 0] - cy, true_coords[:, 1] - cx)
        matched_tid = true_ids[int(np.argmin(d))]
        true_positive = matched_tid in MARKER_POSITIVE
        classification_records.append((t, predicted_positive, true_positive))

records = np.array([(p, t) for _, p, t in classification_records])
classification_accuracy = (records[:, 0] == records[:, 1]).mean()
print(f"classified {len(records)} detections across {N_FRAMES} frames")
print(f"marker-positive/-negative classification accuracy vs. ground truth: "
      f"{classification_accuracy:.1%}")

🧪 **Ground-truth check.** Because every detection is matched to the exact tracked
cell nearest to it, "was this detection correctly classified" is a yes/no fact, not a judgment
call — and the aggregate accuracy above is computed over every detection in every frame, not one
cherry-picked example.

---

## 5 · Track: linking detections across time with the Hungarian algorithm

Module 14 introduced the linear assignment problem for exactly this task: given detections in
frame *t* and frame *t+1*, find the lowest-total-cost one-to-one matching (by distance) and call
that the frame-to-frame tracking link. We check the result by asking, for every predicted link,
whether it actually connects two detections of the *same* ground-truth cell (known from §4's
nearest-position matching).

In [ ]:
# ── 🧪 Hungarian frame-to-frame tracking, checked against ground-truth identity ──
def match_to_ground_truth(labels, t):
    """{object_id: nearest_true_track_id} for every detection in frame t."""
    centroids = detection_centroids(labels)
    true_positions = active_positions_at(tracks, t)
    true_ids = list(true_positions)
    true_coords = np.array([true_positions[tid] for tid in true_ids])
    out = {}
    for obj_id, (cy, cx) in centroids.items():
        d = np.hypot(true_coords[:, 0] - cy, true_coords[:, 1] - cx)
        out[obj_id] = true_ids[int(np.argmin(d))]
    return out


correct_links, total_links = 0, 0
for t in range(N_FRAMES - 1):
    cent_t = detection_centroids(pred_labels[t])
    cent_t1 = detection_centroids(pred_labels[t + 1])
    ids_t, ids_t1 = list(cent_t), list(cent_t1)
    if not ids_t or not ids_t1:
        continue
    cost = np.array([[np.hypot(cent_t[i][0] - cent_t1[j][0], cent_t[i][1] - cent_t1[j][1])
                       for j in ids_t1] for i in ids_t])
    row_ind, col_ind = hungarian_assignment(cost)
    gt_t, gt_t1 = match_to_ground_truth(pred_labels[t], t), match_to_ground_truth(pred_labels[t + 1], t + 1)
    for r, c in zip(row_ind, col_ind):
        if cost[r, c] > SIZE * 0.1:     # reject implausibly distant "matches" (a sanity check)
            continue
        total_links += 1
        correct_links += gt_t[ids_t[r]] == gt_t1[ids_t1[c]]

tracking_accuracy = correct_links / total_links if total_links else 0.0
print(f"{total_links} accepted frame-to-frame links across {N_FRAMES - 1} frame pairs")
print(f"tracking accuracy (fraction of links connecting the SAME true cell): {tracking_accuracy:.1%}")

🔍 **What just happened?** The distance-rejection step (`cost[r, c] > SIZE * 0.1`) is
the tracking equivalent of Module 27's "verify before trusting" habit: the Hungarian algorithm
will happily produce a lowest-cost matching even when every real candidate is far away (e.g. a
cell exiting the field of view), so a plausibility check on the matched distance keeps those
implausible links from being silently counted. The resulting tracking accuracy is checked, not
estimated, because `match_to_ground_truth` ties every detection back to an exact tracked cell
identity.

---

## 6 · Quantify: spatial statistics and colocalization

With detections classified and tracked, we can finally answer something like the biological
question in §0: are reporter-positive cells spatially clustered relative to reporter-negative
ones, and does the colocalization between the two channels match the known association we built
the data with?

In [ ]:
# ── Spatial statistics: are positive cells clustered relative to negative ones? ──
t_final = N_FRAMES - 1
labels_final = pred_labels[t_final]
gt_final = match_to_ground_truth(labels_final, t_final)
centroids_final = detection_centroids(labels_final)

pos_pts = np.array([centroids_final[i] for i in centroids_final if gt_final[i] in MARKER_POSITIVE])
neg_pts = np.array([centroids_final[i] for i in centroids_final if gt_final[i] not in MARKER_POSITIVE])
print(f"final frame: {len(pos_pts)} positive, {len(neg_pts)} negative detections")

if len(pos_pts) >= 2:
    ce_pos = clark_evans_index(pos_pts, SIZE ** 2)
    print(f"Clark-Evans index among POSITIVE cells: {ce_pos:.2f} "
          f"({'clustered' if ce_pos < 1 else 'dispersed/random'})")

    # 🧪 Is that index actually meaningful, or just small-sample noise? A label-permutation null:
    # randomly relabel the SAME detected positions as "positive" (same count), many times, and
    # see where the observed index falls relative to that null distribution.
    all_pts = np.array(list(centroids_final.values()))
    rng_null = np.random.default_rng(SEED)
    null_ce = np.array([clark_evans_index(all_pts[rng_null.choice(len(all_pts), size=len(pos_pts),
                                                                   replace=False)], SIZE ** 2)
                        for _ in range(200)])
    lo, hi = np.percentile(null_ce, [5, 95])
    inside_null = lo <= ce_pos <= hi
    print(f"label-permutation null (positions fixed, labels reshuffled): "
          f"90% range [{lo:.2f}, {hi:.2f}]")
    print(f"observed index {'IS' if inside_null else 'is NOT'} inside that null range")
if len(pos_pts) and len(neg_pts):
    cross_nn = nearest_neighbor_distances(pos_pts, neg_pts)
    print(f"mean distance from a positive cell to its nearest NEGATIVE neighbor: {cross_nn.mean():.1f} px")

# ── Colocalization between the two channels, over the whole final frame ──
coloc = pearson_colocalization(frames_a[t_final], frames_b[t_final])
print(f"\nPearson colocalization, channel A vs. channel B (final frame): {coloc:.2f}")
print(f"(expected to be positive and modest: channel B's blobs sit exactly on top of a SUBSET "
      f"of channel A's blobs, by construction)")

🔍 **What just happened?** Because cell positions in this simulation were placed
independently of marker status (`tid % 2 == 0` has nothing to do with spatial layout), the "true"
Clark–Evans index for positive cells is 1 — but with only a handful of positive detections in one
frame, the *observed* index can land meaningfully away from 1 purely from small-sample noise,
exactly the kind of null-distribution question Module 18 built `csr_envelope` around. Rather than
trust the single point estimate, the label-permutation null above asks the sharper question
directly: if marker status really were spatially irrelevant, how much would the index vary just
from *which* detections happened to be labeled positive, with their positions held fixed? If the
observed index falls inside that null range, the apparent "clustering" is exactly the small-sample
noise you would expect from a true null — a useful negative control confirming the pipeline is not
finding spatial structure that was never built into the data. The positive Pearson colocalization
is expected for a different reason: wherever a channel-B blob exists, it sits exactly on a
channel-A position by construction, so the two channels are correlated overall even though only a
subset of channel-A objects have a channel-B counterpart.

✏️ **Your turn.** Change `MARKER_POSITIVE` to a *spatially* non-random rule — e.g.
cells in the left half of the frame (`tid`'s *initial x position* `< SIZE/2`) — and re-run §1, §4
and this section. Does the Clark–Evans index among positive cells now correctly pick up the
clustering you built in?

---

## 7 · Report: one checked results summary

A real analysis ends with a written summary a biologist can act on — not a notebook's worth of
intermediate numbers. Every number quoted below was computed, and checked against ground truth,
above; nothing here is new.

In [ ]:
# ── The capstone's actual "answer" ──
print("CAPSTONE PIPELINE REPORT")
print("=" * 60)
print(f"Frames analyzed                         : {N_FRAMES}")
print(f"Mean segmentation Dice vs. ground truth  : {np.mean(dices):.3f}")
print(f"Marker classification accuracy           : {classification_accuracy:.1%}")
print(f"Frame-to-frame tracking accuracy         : {tracking_accuracy:.1%}")
if len(pos_pts) >= 2:
    print(f"Spatial pattern of positive cells (final frame): Clark-Evans = {ce_pos:.2f}, "
          f"{'inside' if inside_null else 'OUTSIDE'} the 90% label-permutation null range "
          f"[{lo:.2f}, {hi:.2f}] -> {'no real clustering detected' if inside_null else 'possible real clustering'}")
print(f"Channel A / channel B colocalization      : {coloc:.2f}")
print("=" * 60)
print("All figures above are measured against known ground truth, because every image, label, "
      "track and marker-positive assignment in this notebook was generated by code.")

---

## 8 · Project studio: three extension tracks

Pick one (or more) of these to extend the pipeline above. Each is deliberately scoped to be
doable in an extra session or two, reusing tools you already have.

### 🧮 Track A — classical-CV-heavy: swap the segmenter
Replace `watershed_labels` in §3 with a from-scratch pipeline combining Module 10's tools: Otsu-
style or EM-based thresholding (`em_gaussian_mixture_1d`) for a foreground mask, then a graph-cut
or active-contour (`chan_vese_step`) refinement of each connected component. Compare the resulting
Dice scores, frame by frame, to watershed's — does one approach handle touching/dividing cells
(the `lineage` events) more gracefully than the other?

### 🧠 Track B — deep-learning-heavy: swap in a foundation-model-style segmenter
Train a tiny U-Net-style segmenter in the style of Module 11 (or reuse Module 20's foundation-
model ideas conceptually) on a handful of `simulate_nuclei_image` examples, then apply it in place
of `watershed_labels` in §3. Quantify the accuracy *gap* to watershed's classical result — and
separately, the gap when the deep segmenter is applied with no retraining to a harder noise
condition (Module 26's domain-shift lesson, applied to this pipeline's own segmentation step).

### 🔬 Track C — research-frontier: add a batch effect and virtual staining step
Before segmentation, apply Module 22-style batch effects (a per-frame brightness/contrast shift
simulating a degrading light source over the movie) to channel A, and replace channel B with a
Module 21-style *virtually stained* prediction from channel A instead of a real second channel.
Re-run the full pipeline and measure how far the final Dice, classification and tracking numbers
drift from this notebook's checked baseline — a direct, quantified answer to "how much does
plugging in realistic frontier-model components change what the pipeline reports?"

---

## 9 · Reproducibility checklist

Before trusting (or sharing) any pipeline's output — this one included — check:

- [ ] **Is every source of randomness seeded?** (`SEED` is defined once, at the top, and every
  simulator call traces back to it — confirm by re-running this notebook and checking the numbers
  in §7 come back identical.)
- [ ] **Is there any train/test leakage?** This pipeline has no train/test split in the usual
  sense, but the *principle* still applies: §4's classification threshold was not tuned by peeking
  at `MARKER_POSITIVE` for this specific dataset, and §5's distance-rejection cutoff was chosen
  before looking at the tracking-accuracy result it produces (Module 26).
- [ ] **Is it clearly reported what is simulated vs. real?** Every number in §7 is clearly a
  property of simulated data with known ground truth — restate that explicitly whenever you reuse
  this pipeline's code on real microscopy data, where none of these ground-truth comparisons will
  be available the same way.
- [ ] **What would change on real data?** At minimum: no exact ground-truth mask to compute Dice
  against (you would need manual annotations or a held-out validated reference instead); genuine
  domain shift between your microscope and whatever the segmenter/classifier was tuned on
  (Module 26); and messier, non-independent noise (real cameras, real illumination drift) in place
  of this notebook's clean, independent Gaussian noise model.

---

## 10 · 📘 Lecture notes — the whole course's throughline

Every stage of this capstone traces back to one specific earlier module: §2 restoration to
Module 17 (and 16's noise models); §3 segmentation to Module 10 (and 11, if you took Track B);
§4 detection/classification to Modules 07–08 and 12; §5 tracking to Module 14; §6 spatial
statistics and colocalization to Module 18; and the research-frontier extension tracks in §8
to Modules 20–23. The single idea connecting all of it, repeated at every stage of this notebook,
is the one Module 26 made explicit: a pipeline's output is only as trustworthy as the checks
placed between its steps — not because the tools are unreliable, but because *any* tool, however
well validated in isolation, can be fed an input it was never meant to see. Building the habit of
checking (against ground truth where you have it, against a plausibility range where you do not)
is, more than any single algorithm in this course, the actual skill a working bioimage analyst
needs.

## 📌 Key takeaways

- A real bioimage-analysis pipeline is a chain of checked steps, not a single model — restoring,
  segmenting, classifying, tracking and quantifying each depend on the step before having been
  verified, not merely assumed to have worked.
- Simulated data with known ground truth is what makes every one of those checks possible in this
  course; on real data, the equivalent checks require deliberate effort (manual annotation,
  held-out validated references, plausibility bounds) rather than coming for free.
- The three project-studio tracks are not equally hard, but they are equally legitimate ways to
  extend this course: classical computer vision, deep learning, and integrating research-frontier
  components are three different, complementary directions the same underlying pipeline can grow
  in.
- A reproducibility checklist is worth running on every pipeline you build, including (especially)
  ones that already look like they are "working."

## ✅ Checkpoint

**1. Why does the tracking step (§5) reject frame-to-frame links whose matched distance exceeds
`SIZE * 0.1`, instead of accepting whatever the Hungarian algorithm returns?**

<details><summary>Answer</summary>

The Hungarian algorithm finds the lowest-total-cost *complete* matching between two sets — it
will pair up every detection in frame t with some detection in frame t+1 even if the true match
left the field of view or a new cell just entered, producing a link that is cheapest-available but
still implausible. The distance cutoff is a plausibility check (the same idea as Module 27's
agent verifying its segmentation count) that keeps those forced-but-wrong matches from being
silently counted as correct tracking.
</details>

**2. The final report (§7) states a Clark–Evans index for marker-positive cells that is not
exactly 1, yet §6 concludes there is no real clustering. How can both be true at once?**

<details><summary>Answer</summary>

A point estimate from a handful of detections in one frame will almost never land exactly on its
theoretical value just from sampling noise — the real question is whether the deviation is
*larger* than sampling noise alone would produce. The label-permutation null in §6 answers exactly
that: it repeatedly reshuffles which of the same fixed detected positions count as "positive" and
recomputes the index each time, producing the range of values pure chance would produce if marker
status really were spatially irrelevant (which, by construction — `tid % 2 == 0` — it is). If the
observed index falls inside that null range, the non-1 point estimate is exactly the kind of noise
the null predicts, not evidence of real clustering — the same reasoning Module 18's `csr_envelope`
uses for complete spatial randomness, applied here to a label-permutation question instead.
</details>

**3. Which of the three project-studio tracks would you expect to change the segmentation Dice
score the most, and why?**

<details><summary>Answer</summary>

Track C (batch effects + virtual staining) would likely change it the most, because it alters the
*input* to segmentation itself (a brightness/contrast shift is exactly the kind of domain shift
Module 26 showed can sharply reduce a model's accuracy), whereas Track A and Track B change which
*algorithm* processes the same, unshifted input. That said, this is a prediction to check, not a
fact to take on faith — rerunning the pipeline is the only way to know for a specific choice of
shift magnitude and segmenter.
</details>

## 🧠 Questions & discussion

- This pipeline's final report (§7) is five numbers. If you were handing this report to a
  biologist collaborator who had never seen the pipeline's code, what else would you want to tell
  them before they acted on those five numbers?
- Every ground-truth check in this notebook was possible because the data is simulated. Pick one
  check (segmentation Dice, classification accuracy, or tracking accuracy) and describe concretely
  what you would do instead to estimate the same quantity on a real dataset, where no simulator
  ground truth exists.
- The three project-studio tracks represent three different kinds of "improvement": classical
  algorithmic refinement, deep-learning capacity, and integrating newer research methods. If you
  only had time for one, which would you pick for a pipeline headed into a real, ongoing
  experiment (as opposed to a one-off analysis), and why?

## 🚀 Bonus — going further

- Add a fourth channel and fourth ground-truth rule (e.g. cell-cycle phase, correlated with time
  since the cell's last division via `lineage`) and extend §4's classification step to a 3-class
  problem instead of binary positive/negative.
- Replace the fixed `marker_threshold` in §4 with an EM-based threshold (`em_gaussian_mixture_1d`
  from Module 10) fit per frame, and compare classification accuracy to the fixed-threshold
  version — does an adaptive threshold help more in early frames (fewer cells, sparser signal) or
  late frames (more cells, more division noise)?
- Turn §9's reproducibility checklist into a short automated script (in the spirit of
  `validate_notebooks.py`) that checks a notebook defines `SEED`, states which data are simulated,
  and does not silently reuse a tuning sample as its reported test result.

## 📝 Lab note

Record, in your own words:

1. The five numbers from §7's report, and one sentence on which one you trust least and why.
2. One sentence naming which project-studio track (§8) you would pursue first, and what result you
   predict before running it.
3. One item from the reproducibility checklist (§9) that this pipeline satisfies trivially because
   the data is simulated, and what you would need to add to satisfy it on real data.

## 📚 References

This capstone deliberately reuses tools and ideas from across the whole course rather than
introducing new primary sources; see each earlier module's own References section for full
citations. Three references specific to the project-studio tracks and reproducibility checklist:

1. Walsh, I. et al. (2021). DOME: recommendations for supervised machine learning validation in
   biology. *Nature Methods*, 18 (the basis for §9's reproducibility checklist).
2. Çiçek, Ö. et al. (2016) and Ronneberger, O. et al. (2015) — the 3D and 2D U-Net papers
   underlying Track B's deep-learning segmenter (see Modules 11 and 25's references for full
   citations).
3. Ouyang, W. et al. (2022). BioImage.IO... (see Module 27's references) — relevant to extending
   Track C with a real pretrained virtual-staining or foundation-model component instead of this
   notebook's from-scratch versions.

## ➡️ Next

That's the course. From here: revisit `FRONTIER_MAP.md` for a map of which research-frontier
module to go deeper into next, or `STUDY_GUIDE.md` for how the 29 modules connect back to the
CMU 02-740 course materials this track was built to complement.